In [0]:
import pytest
import sys
import os
from pathlib import Path

Define the path to the repository root

In [0]:
sys.dont_write_bytecode = True
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
workspace_path = notebook_path if notebook_path.startswith("/Workspace/") else "/Workspace" + notebook_path

In [0]:
repo_root = str(Path(workspace_path).parent.parent)

Add the root to sys.path for correct src imports

In [0]:
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

Change the working directory so pytest finds the tests/ folder

In [0]:
os.chdir(repo_root)

Run pytest with verbose output (-v)

In [0]:
print(f"Running tests in directory: {repo_root}")
exit_code = pytest.main(["-v", "-p", "no:cacheprovider", "tests/"])

if exit_code == 0:
    print("\n✅ All tests passed successfully!")
else:
    print(f"\n❌ Tests failed with exit code: {exit_code}. Review the details above.")

## Validation rules

This section runs post-ingestion verification queries directly against the published `tpch_silver` Delta tables to verify that the three core data validation rules are strictly upheld:
1. **Temporal Integrity:** No shipment leaves before order placement, and no item is received before shipment.
2. **Categorical Allowed Lists:** `ship_mode`, `return_flag`, and `line_status` contain only expected whitelist values.
3. **Referential Integrity:** No orphaned line items, and no orders exist with zero items.

In [0]:
from pyspark.sql import functions as F
from config import env

os.environ["DB_CATALOG"] = "workspace"
os.environ["SCHEMA_PREFIX"] = "tpch"

silver_orders = spark.table("workspace.tpch_silver.orders")
silver_lineitem = spark.table("workspace.tpch_silver.lineitem")

print(f"Auditing Silver tables in schema: {env.CATALOG}.{env.SILVER_SCHEMA}")

### 1. Temporal Integrity Audit
Validating that:
- `ship_date >= order_date`
- `receipt_date >= ship_date`

In [0]:
# Join lineitem with orders to compare order_date and ship_date
joined_dates = silver_lineitem.join(
    silver_orders,
    silver_lineitem.order_key == silver_orders.order_key,
    "inner"
)

# Rule 1a: ship_date < order_date
invalid_ship_dates = joined_dates.filter(F.col("ship_date") < F.col("order_date")).count()

# Rule 1b: receipt_date < ship_date
invalid_receipt_dates = silver_lineitem.filter(F.col("receipt_date") < F.col("ship_date")).count()

print(f"Rule 1a Violation (ship_date < order_date): {invalid_ship_dates} records")
print(f"Rule 1b Violation (receipt_date < ship_date): {invalid_receipt_dates} records")

assert invalid_ship_dates == 0, f"Temporal violation: {invalid_ship_dates} records shipped before order date!"
assert invalid_receipt_dates == 0, f"Temporal violation: {invalid_receipt_dates} records received before ship date!"
print(" Temporal integrity verified.")

### 2. Categorical Allowed Lists Audit
Ensuring categorical dimensions strictly match the profiled whitelist.

In [0]:
allowed_ship_modes = {"AIR", "FOB", "MAIL", "RAIL", "REG AIR", "SHIP", "TRUCK"}
allowed_return_flags = {"N", "R", "A"}
allowed_line_statuses = {"O", "F"}

# Distinct values present in Silver
actual_ship_modes = {row.ship_mode for row in silver_lineitem.select("ship_mode").distinct().collect()}
actual_return_flags = {row.return_flag for row in silver_lineitem.select("return_flag").distinct().collect()}
actual_line_statuses = {row.line_status for row in silver_lineitem.select("line_status").distinct().collect()}

# Check unexpected values
unexpected_ship_modes = actual_ship_modes - allowed_ship_modes
unexpected_return_flags = actual_return_flags - allowed_return_flags
unexpected_line_statuses = actual_line_statuses - allowed_line_statuses

print(f"Unexpected ship modes: {unexpected_ship_modes or 'None'}")
print(f"Unexpected return flags: {unexpected_return_flags or 'None'}")
print(f"Unexpected line statuses: {unexpected_line_statuses or 'None'}")

assert not unexpected_ship_modes, f"Disallowed ship modes found: {unexpected_ship_modes}"
assert not unexpected_return_flags, f"Disallowed return flags found: {unexpected_return_flags}"
assert not unexpected_line_statuses, f"Disallowed line statuses found: {unexpected_line_statuses}"
print(" Categorical allowed lists verified.")

### 3. Referential Integrity Audit
Checking that every lineitem links to an existing order, and no orders exist without associated lineitems.

In [0]:
# Orphaned lineitems (lineitem with no matching order)
orphaned_lineitems = silver_lineitem.join(
    silver_orders,
    silver_lineitem.order_key == silver_orders.order_key,
    "left_anti"
).count()

# Empty orders (orders with no items)
orders_without_items = silver_orders.join(
    silver_lineitem,
    silver_orders.order_key == silver_lineitem.order_key,
    "left_anti"
).count()

print(f"Orphaned lineitems count: {orphaned_lineitems}")
print(f"Orders without items count: {orders_without_items}")

assert orphaned_lineitems == 0, f"Referential error: {orphaned_lineitems} orphaned lineitems found!"
assert orders_without_items == 0, f"Referential error: {orders_without_items} orders without line items found!"
print(" Referential integrity verified.")